# GKG A/B Benchmark

**Arm A** — raw LLM, single prompt.

**Arm B** — GKG: graph structure (Macro→Meso→Micro) used as blueprint context for coder LLM.

Two evaluation axes:
1. **Graph structure quality** (B only) — does the architecture make sense?
2. **Final code quality** (A vs B) — does the output parse, have symbols, respect constraints?

In [2]:
import ast, re, time, json, statistics
from dataclasses import dataclass, field
from typing import Callable
from IPython.display import display, HTML, Markdown

from gkg import (
    Graph, Level, Status, Phase, EdgeKind, Ownership, DesignPattern,
    MesoPayload, MicroPayload, MacroPayload,
)
from commands import dispatch, describe_state, describe_for_coder, run_agent
from ollama_client import OllamaClient

MODEL = 'qwen2.5-coder:1.5b'
client = OllamaClient(MODEL)
try:
    pong = client.complete('reply with the single word: pong', max_tokens=10)
    print(f'ollama ok: {pong.strip()!r}')
except Exception as e:
    print(f'ollama not reachable: {e}\nStart ollama and re-run.')

ollama ok: 'pong'


## 1 · Tasks

4 tasks testing multi-module architecture, design-pattern enforcement, and ownership constraints. Each defines a raw spec (Arm A) and a graph builder (Arm B).

In [3]:
@dataclass
class Check:
    name: str
    fn: Callable[[str], bool]
    weight: float = 1.0

@dataclass
class Task:
    id: str
    title: str
    spec: str                                    # raw prompt for Arm A
    build_graph: Callable[[OllamaClient], Graph] # builds MACRO/MESO/MICRO graph
    expected_symbols: list[str]
    forbidden: list[str]
    checks: list[Check] = field(default_factory=list)
    goal: str = ''  # architect agent natural-language goal (Arm C)


# ── T4: EventBus (OBSERVER pattern) ───────────────────────────
def build_t4(client):
    g = Graph()
    m = g.add_macro(name='Events', intent='event bus module')
    g.promote(m, Status.LAYER_STABLE)
    g.advance_phase(Phase.MACRO_STABLE)
    meso = g.add_meso(parent=m, name='EventBus', intent='pub/sub event bus',
                      design_pattern=DesignPattern.OBSERVER,
                      behaviors=['subscribe', 'unsubscribe', 'notify', 'publish'])
    g.promote(meso, Status.LAYER_STABLE)
    g.advance_phase(Phase.MESO_STABLE)
    m_sub   = g.add_micro(parent=meso, name='subscribe_impl',
                          intent='register handler callable for event_type in internal dict',
                          inputs=['event_type', 'handler'], outputs=[])
    m_unsub = g.add_micro(parent=meso, name='unsubscribe_impl',
                          intent='remove handler for event_type if registered',
                          inputs=['event_type', 'handler'], outputs=[])
    m_notif = g.add_micro(parent=meso, name='notify_impl',
                          intent='call every handler registered for event_type with data',
                          inputs=['event_type', 'data'], outputs=[])
    m_pub   = g.add_micro(parent=meso, name='publish_impl',
                          intent='alias for notify — dispatch event to all subscribers',
                          inputs=['event_type', 'data'], outputs=[])
    g.promote_group([m_sub, m_unsub, m_notif, m_pub], Status.LAYER_STABLE)
    g.advance_phase(Phase.MICRO_STABLE)
    g.promote_group([m_sub, m_unsub, m_notif, m_pub], Status.READY)
    return g

TASK4 = Task(
    id='t4', title='EventBus (OBSERVER)',
    spec=(
        'Write a Python class EventBus implementing a pub/sub event system. '
        '__init__(self) creates an empty dict mapping event_type strings to lists of handler callables. '
        'subscribe(self, event_type, handler) appends handler to the list for event_type. '
        'unsubscribe(self, event_type, handler) removes handler from the list if present. '
        'notify(self, event_type, data) calls each registered handler with data. '
        'publish(self, event_type, data) is an alias that calls notify. '
        'Do not use eval, exec, or threading.'
    ),
    build_graph=build_t4,
    expected_symbols=['EventBus', 'subscribe', 'unsubscribe', 'notify', 'publish'],
    forbidden=['eval(', 'exec(', 'threading'],
    checks=[
        Check('is_class',       lambda c: 'class EventBus' in c, 2),
        Check('has_subscribe',  lambda c: 'def subscribe' in c),
        Check('has_notify',     lambda c: 'def notify' in c),
        Check('has_publish',    lambda c: 'def publish' in c),
        Check('uses_dict',      lambda c: 'dict(' in c or ': []' in c or '= {}' in c or 'defaultdict' in c),
        Check('calls_handlers', lambda c: 'for ' in c and '(' in c),
    ],
    goal=(
        'Design a GKG for an EventBus pub/sub system in Python. '
        'One MODULE named Events. '
        'One CLASS: EventBus with OBSERVER pattern, behaviors: subscribe, unsubscribe, notify, publish. '
        'Four MICRO methods: subscribe(event_type, handler), unsubscribe(event_type, handler), '
        'notify(event_type, data), publish(event_type, data). publish delegates to notify.'
    ),
)


# ── T5: HTTP Router (COMPOSITE + STRATEGY, multi-MACRO) ──────────
def build_t5(client):
    g = Graph()
    m_router = g.add_macro(name='RouterModule', intent='URL routing module')
    m_mw     = g.add_macro(name='MiddlewareModule', intent='middleware chain module')
    g.promote_group([m_router, m_mw], Status.LAYER_STABLE)
    g.advance_phase(Phase.MACRO_STABLE)
    meso_r = g.add_meso(parent=m_router, name='Router',
                        intent='composite router mapping path strings to handler callables',
                        design_pattern=DesignPattern.COMPOSITE,
                        behaviors=['add', 'remove', 'match'])
    meso_m = g.add_meso(parent=m_mw, name='MiddlewareChain',
                        intent='ordered middleware pipeline — strategy pattern',
                        design_pattern=DesignPattern.STRATEGY,
                        behaviors=['use', 'execute'])
    g.promote_group([meso_r, meso_m], Status.LAYER_STABLE)
    g.add_edge(meso_r, meso_m, EdgeKind.DEPENDS_ON)   # Router depends on MiddlewareChain
    g.advance_phase(Phase.MESO_STABLE)
    mi_add   = g.add_micro(parent=meso_r, name='add_route',
                           intent='register handler callable for path string in routes dict',
                           inputs=['path', 'handler'], outputs=[])
    mi_match = g.add_micro(parent=meso_r, name='match',
                           intent='return handler for path or None if not registered',
                           inputs=['path'], outputs=['handler | None'])
    mi_use   = g.add_micro(parent=meso_m, name='use',
                           intent='append middleware callable to chain list',
                           inputs=['middleware'], outputs=[])
    mi_exec  = g.add_micro(parent=meso_m, name='execute',
                           intent='pass request dict through each middleware in order; return result',
                           inputs=['request'], outputs=['response'])
    g.add_edge(mi_match, mi_exec, EdgeKind.CALLS, order=1)  # match result flows to execute
    g.promote_group([mi_add, mi_match, mi_use, mi_exec], Status.LAYER_STABLE)
    g.advance_phase(Phase.MICRO_STABLE)
    g.promote_group([mi_add, mi_match, mi_use, mi_exec], Status.READY)
    return g

TASK5 = Task(
    id='t5', title='HTTP Router (COMPOSITE+STRATEGY)',
    spec=(
        'Write two Python classes: Router and MiddlewareChain. '
        'Router.__init__(self) creates an empty dict for routes. '
        'Router.add_route(self, path, handler) registers a callable handler for a path string. '
        'Router.remove(self, path) deletes path from routes if present. '
        'Router.match(self, path) returns the handler callable if path is registered, else None. '
        'MiddlewareChain.__init__(self) creates an empty list for middleware callables. '
        'MiddlewareChain.use(self, middleware) appends a middleware callable to the chain. '
        'MiddlewareChain.execute(self, request) passes a request dict through each middleware in order and returns the result. '
        'Do not use eval or exec.'
    ),
    build_graph=build_t5,
    expected_symbols=['Router', 'MiddlewareChain', 'add_route', 'match', 'use', 'execute'],
    forbidden=['eval(', 'exec('],
    checks=[
        Check('has_Router',          lambda c: 'class Router' in c, 2),
        Check('has_MiddlewareChain', lambda c: 'class MiddlewareChain' in c, 2),
        Check('has_add_route',       lambda c: 'def add_route' in c),
        Check('has_match',           lambda c: 'def match' in c),
        Check('has_execute',         lambda c: 'def execute' in c),
        Check('match_returns_none',  lambda c: 'return None' in c or '.get(' in c),
        Check('mw_loops',            lambda c: 'for ' in c),
    ],
    goal=(
        'Design a GKG for an HTTP router system in Python. Two modules. '
        'MODULE 1: RouterModule. CLASS: Router (COMPOSITE pattern), '
        'behaviors: add, remove, match. MICRO methods: add_route(path, handler), remove(path), '
        'match(path) returns handler or None. '
        'MODULE 2: MiddlewareModule. CLASS: MiddlewareChain (STRATEGY pattern), '
        'behaviors: use, execute. MICRO methods: use(middleware), execute(request) returns response. '
        'Add DEPENDS_ON edge from Router to MiddlewareChain. '
        'Add CALLS edge (order=1) from match to execute.'
    ),
)


# ── T6: UserRepository (REPOSITORY pattern) ──────────────────
def build_t6(client):
    g = Graph()
    m_store  = g.add_macro(name='Storage', intent='persistence layer')
    m_models = g.add_macro(name='Models',  intent='data model definitions')
    g.promote_group([m_store, m_models], Status.LAYER_STABLE)
    g.advance_phase(Phase.MACRO_STABLE)
    # Models module: defines User schema
    meso_model = g.add_meso(parent=m_models, name='UserModel',
                            intent='User data structure — dict with id, name, email fields',
                            design_pattern=DesignPattern.FACTORY,
                            behaviors=['create', 'from_dict'])
    # Storage module: CRUD repository depending on UserModel
    meso_repo = g.add_meso(parent=m_store, name='UserRepository',
                           intent='in-memory CRUD repository for User records',
                           design_pattern=DesignPattern.REPOSITORY,
                           behaviors=['find', 'save', 'delete', 'update'])
    g.promote_group([meso_model, meso_repo], Status.LAYER_STABLE)
    g.add_edge(meso_repo, meso_model, EdgeKind.DEPENDS_ON)
    g.advance_phase(Phase.MESO_STABLE)
    # UserModel micros
    mi_create = g.add_micro(parent=meso_model, name='create',
                            intent='return dict with keys id (str), name (str), email (str)',
                            inputs=['id', 'name', 'email'], outputs=['User dict'])
    # UserRepository micros
    mi_find = g.add_micro(parent=meso_repo, name='find_by_id',
                          intent='return user dict if user_id key exists in store, else None',
                          inputs=['user_id'], outputs=['User | None'])
    mi_save = g.add_micro(parent=meso_repo, name='save',
                          intent='insert or overwrite user dict using user["id"] as key; return True',
                          inputs=['user'], outputs=['bool'])
    mi_del  = g.add_micro(parent=meso_repo, name='delete',
                          intent='remove entry by user_id; return True if existed else False',
                          inputs=['user_id'], outputs=['bool'])
    mi_upd  = g.add_micro(parent=meso_repo, name='update',
                          intent='merge fields dict into existing record; return False if not found else True',
                          inputs=['user_id', 'fields'], outputs=['bool'])
    g.promote_group([mi_create, mi_find, mi_save, mi_del, mi_upd], Status.LAYER_STABLE)
    g.advance_phase(Phase.MICRO_STABLE)
    g.promote_group([mi_create, mi_find, mi_save, mi_del, mi_upd], Status.READY)
    return g

TASK6 = Task(
    id='t6', title='UserRepository (REPOSITORY)',
    spec=(
        'Write a Python class UserRepository implementing an in-memory CRUD store. '
        '__init__(self) creates an empty dict as the internal store. '
        'find_by_id(self, user_id) returns the user dict if found, else None. '
        'save(self, user) stores user dict using user["id"] as key, returns True. '
        'delete(self, user_id) removes the entry by user_id, returns True if existed else False. '
        'update(self, user_id, fields) merges fields dict into existing record via dict.update(), '
        'returns False if user_id not found else True. '
        'Do not use eval, exec, or an external database. Use only a dict.'
    ),
    build_graph=build_t6,
    expected_symbols=['UserRepository', 'UserModel', 'find_by_id', 'save', 'delete', 'update', 'create'],
    forbidden=['eval(', 'exec('],
    checks=[
        Check('has_UserRepository', lambda c: 'class UserRepository' in c, 2),
        Check('has_UserModel',      lambda c: 'class UserModel' in c or 'UserModel' in c, 2),
        Check('has_find',           lambda c: 'def find_by_id' in c),
        Check('has_save',           lambda c: 'def save' in c),
        Check('has_delete',         lambda c: 'def delete' in c),
        Check('has_update',         lambda c: 'def update' in c),
        Check('has_create',         lambda c: 'def create' in c),
        Check('returns_none',       lambda c: 'return None' in c),
        Check('returns_bool',       lambda c: 'return True' in c and 'return False' in c),
    ],
    goal=(
        'Design a GKG for a user repository system in Python. Two modules. '
        'MODULE 1: Models. CLASS: UserModel (FACTORY pattern), '
        'behaviors: create, from_dict. MICRO method: create(id, name, email) returns User dict. '
        'MODULE 2: Storage. CLASS: UserRepository (REPOSITORY pattern), '
        'behaviors: find, save, delete, update. '
        'MICRO methods: find_by_id(user_id) returns User or None, save(user) returns bool, '
        'delete(user_id) returns bool, update(user_id, fields) returns bool. '
        'Add DEPENDS_ON edge from UserRepository to UserModel.'
    ),
)


# ── T7: Producer-Consumer (ownership stress test) ────────────
def build_t7(client):
    g = Graph()
    m_q    = g.add_macro(name='Queue',    intent='thread-safe bounded queue',
                         ownership=Ownership.MULTI_SYNCED)
    m_prod = g.add_macro(name='Producer', intent='item producer module',
                         ownership=Ownership.SINGLE_WRITER)
    m_cons = g.add_macro(name='Consumer', intent='item consumer module',
                         ownership=Ownership.SINGLE_WRITER)
    g.promote_group([m_q, m_prod, m_cons], Status.LAYER_STABLE)
    g.advance_phase(Phase.MACRO_STABLE)
    meso_q    = g.add_meso(parent=m_q,    name='BoundedQueue',
                           intent='fixed-capacity FIFO queue backed by collections.deque',
                           behaviors=['enqueue', 'dequeue'])
    meso_prod = g.add_meso(parent=m_prod, name='ProducerAgent',
                           intent='generates items and pushes to BoundedQueue',
                           behaviors=['run', 'produce'])
    meso_cons = g.add_meso(parent=m_cons, name='ConsumerAgent',
                           intent='pulls items from BoundedQueue and processes them',
                           behaviors=['run', 'consume'])
    g.promote_group([meso_q, meso_prod, meso_cons], Status.LAYER_STABLE)
    g.add_edge(meso_prod, meso_q, EdgeKind.DEPENDS_ON)  # ProducerAgent depends on BoundedQueue
    g.add_edge(meso_cons, meso_q, EdgeKind.DEPENDS_ON)  # ConsumerAgent depends on BoundedQueue
    g.advance_phase(Phase.MESO_STABLE)
    mi_enq  = g.add_micro(parent=meso_q,    name='enqueue',
                          intent='append item to deque if len < maxsize; return True else False',
                          inputs=['item'], outputs=['bool'])
    mi_deq  = g.add_micro(parent=meso_q,    name='dequeue',
                          intent='popleft and return item if deque non-empty, else return None',
                          inputs=[], outputs=['item | None'])
    mi_prun = g.add_micro(parent=meso_prod, name='run',
                          intent='call produce() n times; push each result via queue.enqueue()',
                          inputs=['queue', 'n'], outputs=[])
    mi_crun = g.add_micro(parent=meso_cons, name='run',
                          intent='call queue.dequeue() n times; call process(item) if item is not None',
                          inputs=['queue', 'n'], outputs=[])
    g.promote_group([mi_enq, mi_deq], Status.LAYER_STABLE)
    g.promote_group([mi_prun, mi_crun], Status.LAYER_STABLE)
    g.advance_phase(Phase.MICRO_STABLE)
    g.promote_group([mi_enq, mi_deq, mi_prun, mi_crun], Status.READY)
    return g

TASK7 = Task(
    id='t7', title='Producer-Consumer (ownership)',
    spec=(
        'Write three Python classes: BoundedQueue, ProducerAgent, ConsumerAgent. '
        'BoundedQueue.__init__(self, maxsize) stores maxsize and creates a collections.deque. '
        'BoundedQueue.enqueue(self, item) appends item if len(deque) < maxsize and returns True, else returns False. '
        'BoundedQueue.dequeue(self) popleft and returns item if deque non-empty, else returns None. '
        'ProducerAgent.__init__(self, source) stores source callable. '
        'ProducerAgent.run(self, queue, n) calls source() n times, pushing each result via queue.enqueue(). '
        'ConsumerAgent.__init__(self, process) stores process callable. '
        'ConsumerAgent.run(self, queue, n) calls queue.dequeue() n times, '
        'calling process(item) when item is not None. '
        'Do not use eval, exec, threading, or asyncio. Import collections.'
    ),
    build_graph=build_t7,
    expected_symbols=['BoundedQueue', 'ProducerAgent', 'ConsumerAgent', 'enqueue', 'dequeue', 'run'],
    forbidden=['eval(', 'exec(', 'threading', 'asyncio'],
    checks=[
        Check('has_BoundedQueue',   lambda c: 'class BoundedQueue' in c, 2),
        Check('has_ProducerAgent',  lambda c: 'class ProducerAgent' in c),
        Check('has_ConsumerAgent',  lambda c: 'class ConsumerAgent' in c),
        Check('has_enqueue',        lambda c: 'def enqueue' in c),
        Check('has_dequeue',        lambda c: 'def dequeue' in c),
        Check('uses_deque',         lambda c: 'deque' in c),
        Check('enqueue_bool',       lambda c: 'return True' in c and 'return False' in c),
        Check('dequeue_none',       lambda c: 'return None' in c),
        Check('run_loops',          lambda c: 'for ' in c or 'range(' in c),
    ],
    goal=(
        'Design a GKG for a producer-consumer system in Python. Three modules. '
        'MODULE 1: Queue (MULTI_SYNCED ownership). CLASS: BoundedQueue, '
        'behaviors: enqueue, dequeue. MICRO methods: enqueue(item) returns bool, '
        'dequeue() returns item or None. '
        'MODULE 2: Producer (SINGLE_WRITER). CLASS: ProducerAgent, '
        'behaviors: run, produce. MICRO method: run(queue, n). '
        'MODULE 3: Consumer (SINGLE_WRITER). CLASS: ConsumerAgent, '
        'behaviors: run, consume. MICRO method: run(queue, n). '
        'Add DEPENDS_ON edges from ProducerAgent to BoundedQueue '
        'and from ConsumerAgent to BoundedQueue.'
    ),
)

TASKS = [TASK4, TASK5, TASK6, TASK7]
print(f'{len(TASKS)} tasks')


4 tasks


## 2 · Scoring

In [4]:
@dataclass
class ScoreCard:
    parses: float = 0          # 2 pts
    symbols_found: int = 0     # 2 pts (proportional)
    symbols_total: int = 0
    forbidden_clean: float = 0 # 1 pt
    check_scores: dict = field(default_factory=dict)
    total: float = 0
    max_total: float = 0

@dataclass
class GraphScore:
    """Evaluates graph structure — Macro/Meso/Micro."""
    has_macro: bool = False
    has_meso: bool = False
    has_micro: bool = False
    level_depth: int = 0               # zoom levels populated (1-3)
    meso_has_behaviors: bool = False
    micro_has_contracts: bool = False  # inputs/outputs declared
    all_macros_populated: bool = False # every MACRO has >=1 MESO child
    cross_module_edges: bool = False   # edges span different MACRO subtrees
    validation_clean: bool = False
    total: float = 0
    max_total: float = 9  # depth(3)+behaviors(1)+contracts(1)+populated(1)+edges(1)+valid(2)


def score_code(code: str, task: Task) -> ScoreCard:
    sc = ScoreCard(symbols_total=len(task.expected_symbols))
    try:
        tree = ast.parse(code)
        sc.parses = 1
        names = set()
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                names.add(node.name)
            elif isinstance(node, ast.ClassDef):
                names.add(node.name)
        sc.symbols_found = sum(1 for s in task.expected_symbols if s in names)
    except SyntaxError:
        pass
    sc.forbidden_clean = 1 if not any(f in code for f in task.forbidden) else 0
    for ch in task.checks:
        sc.check_scores[ch.name] = ch.weight if ch.fn(code) else 0
    parse_pts = 2 * sc.parses
    symbol_pts = 2 * (sc.symbols_found / max(sc.symbols_total, 1))
    forbidden_pts = sc.forbidden_clean
    check_pts = sum(sc.check_scores.values())
    sc.total = parse_pts + symbol_pts + forbidden_pts + check_pts
    sc.max_total = 2 + 2 + 1 + sum(ch.weight for ch in task.checks)
    return sc


def score_graph(g: Graph) -> GraphScore:
    gs = GraphScore()
    levels_present = set(n.level for n in g.nodes.values())
    gs.has_macro = Level.MACRO in levels_present
    gs.has_meso  = Level.MESO  in levels_present
    gs.has_micro = Level.MICRO in levels_present
    gs.level_depth = sum([gs.has_macro, gs.has_meso, gs.has_micro])

    for n in g.nodes.values():
        if n.level == Level.MESO and isinstance(n.payload, MesoPayload):
            if n.payload.behaviors:
                gs.meso_has_behaviors = True
        if n.level == Level.MICRO and isinstance(n.payload, MicroPayload):
            if n.payload.inputs or n.payload.outputs:
                gs.micro_has_contracts = True

    macros = [n for n in g.nodes.values() if n.level == Level.MACRO]
    gs.all_macros_populated = bool(macros) and all(bool(n.children) for n in macros)

    def _macro_root(nid):
        n = g.nodes.get(nid)
        while n and n.parent:
            n = g.nodes.get(n.parent)
        return n.id if n else nid

    gs.cross_module_edges = bool(g.edges) and any(
        _macro_root(e.src) != _macro_root(e.dst) for e in g.edges.values()
    )
    gs.validation_clean = len(g.validate()) == 0
    gs.total = (gs.level_depth
              + gs.meso_has_behaviors
              + gs.micro_has_contracts
              + gs.all_macros_populated
              + gs.cross_module_edges
              + 2 * gs.validation_clean)
    gs.max_total = 9
    return gs


def extract_code(text: str) -> str:
    m = re.search(r'```(?:python)?\s*\n(.*?)```', text, re.S)
    return m.group(1).strip() if m else text.strip()

print('scoring ready')

scoring ready


## 3 · Run both arms

In [5]:
RAW_SYS = 'You output only a single fenced python code block. No explanation.'

@dataclass
class RunResult:
    arm: str
    task_id: str
    code: str
    code_score: ScoreCard
    graph_score: GraphScore    # only meaningful for B / C
    elapsed: float
    graph: Graph = None


def _coder_prompt(ctx: str) -> str:
    return (
        f"You are a coder agent. The GKG blueprint below defines MODULE/CLASS/METHOD structure.\n"
        f"Use it as your architecture contract: match class names, method signatures, and intents.\n"
        f"Write COMPLETE, WORKING Python code. No pass, no TODO, no placeholders.\n"
        f"Fill every method body with real logic derived from its intent and input/output contracts.\n\n"
        f"GKG BLUEPRINT:\n{ctx}\n\n"
        f"Write only a single fenced python code block."
    )


def _close_graph(g: Graph) -> None:
    try:
        ready = [n.id for n in g.nodes.values()
                 if n.level == Level.MICRO and n.status == Status.READY]
        if ready:
            g.promote_group(ready, Status.CODED)
        if not g.validate():
            g.advance_phase(Phase.DONE)
    except Exception:
        pass


def run_arm_a(task: Task) -> RunResult:
    t0 = time.time()
    raw = client.complete(task.spec, system=RAW_SYS, temperature=0.2)
    code = extract_code(raw)
    return RunResult('A_raw', task.id, code, score_code(code, task),
                     GraphScore(), time.time() - t0)


def run_arm_b(task: Task) -> RunResult:
    t0 = time.time()
    g = task.build_graph(client)
    ctx = describe_for_coder(g)
    raw = client.complete(_coder_prompt(ctx), system=RAW_SYS, temperature=0.2)
    code = extract_code(raw)
    elapsed = time.time() - t0
    graph_score = score_graph(g)
    _close_graph(g)
    return RunResult('B_gkg', task.id, code, score_code(code, task),
                     graph_score, elapsed, g)


def run_arm_c(task: Task) -> RunResult:
    """Arm C: architect agent builds graph from goal, then coder generates code."""
    if not task.goal:
        return RunResult('C_agent', task.id, '', ScoreCard(), GraphScore(), 0.0)
    t0 = time.time()
    g = Graph()
    run_agent(g, client, task.goal, max_steps=50, verbose=False)
    ctx = describe_for_coder(g)
    raw = client.complete(_coder_prompt(ctx), system=RAW_SYS, temperature=0.2)
    code = extract_code(raw)
    elapsed = time.time() - t0
    graph_score = score_graph(g)
    _close_graph(g)
    return RunResult('C_agent', task.id, code, score_code(code, task),
                     graph_score, elapsed, g)


N = 3
all_results: list[RunResult] = []

for task in TASKS:
    print(f'\n\u2501\u2501 {task.title} \u2501\u2501')
    for trial in range(N):
        print(f'  trial {trial+1}/{N}: ', end='', flush=True)
        try:
            ra = run_arm_a(task)
            all_results.append(ra)
            print(f'A={ra.code_score.total:.1f}/{ra.code_score.max_total:.0f} ', end='')
        except Exception as e:
            print(f'A=ERR({e}) ', end='')
            all_results.append(RunResult('A_raw', task.id, '', ScoreCard(), GraphScore(), 0.0))
        try:
            rb = run_arm_b(task)
            all_results.append(rb)
            print(f'B={rb.code_score.total:.1f}/{rb.code_score.max_total:.0f} ', end='')
        except Exception as e:
            print(f'B=ERR({e}) ', end='')
            all_results.append(RunResult('B_gkg', task.id, '', ScoreCard(), GraphScore(), 0.0))
        try:
            rc = run_arm_c(task)
            all_results.append(rc)
            print(f'C={rc.code_score.total:.1f}/{rc.code_score.max_total:.0f} '
                  f'graph={rc.graph_score.total:.0f}/{rc.graph_score.max_total:.0f}')
        except Exception as e:
            print(f'C=ERR({e})')
            all_results.append(RunResult('C_agent', task.id, '', ScoreCard(), GraphScore(), 0.0))

print('\n\u2713 done')



━━ EventBus (OBSERVER) ━━
  trial 1/3: A=10.6/12 B=12.0/12 graph=8/9
  trial 2/3: A=10.6/12 B=12.0/12 graph=8/9
  trial 3/3: A=10.6/12 B=12.0/12 graph=8/9

━━ HTTP Router (COMPOSITE+STRATEGY) ━━
  trial 1/3: A=14.0/14 B=13.3/14 graph=9/9
  trial 2/3: A=14.0/14 B=13.3/14 graph=9/9
  trial 3/3: A=14.0/14 B=13.3/14 graph=9/9

━━ UserRepository (REPOSITORY) ━━
  trial 1/3: A=11.4/16 B=15.0/16 graph=9/9
  trial 2/3: A=11.4/16 B=15.0/16 graph=9/9
  trial 3/3: A=11.4/16 B=15.0/16 graph=9/9

━━ Producer-Consumer (ownership) ━━
  trial 1/3: A=15.0/15 B=15.0/15 graph=8/9
  trial 2/3: A=15.0/15 B=15.0/15 graph=8/9
  trial 3/3: A=15.0/15 B=15.0/15 graph=8/9

✓ done


## 4 · Code quality table

In [6]:
rows = ['| Task | Arm | Mean | Std | Max | Time |']
rows.append('|------|-----|------|-----|-----|------|')
for task in TASKS:
    for arm_id, label in [('A_raw','🟢 Raw'), ('B_gkg','🔵 GKG'), ('C_agent','🟡 Agent')]:
        runs = [r for r in all_results if r.task_id == task.id and r.arm == arm_id]
        if not runs: continue
        tots = [r.code_score.total for r in runs]
        mx = runs[0].code_score.max_total
        rows.append(f'| {task.title} | {label} | {statistics.mean(tots):.1f} | '
                    f'{statistics.pstdev(tots):.1f} | {mx:.0f} | '
                    f'{statistics.mean([r.elapsed for r in runs]):.1f}s |')
display(Markdown('\n'.join(rows)))

| Task | Arm | Mean | Std | Max | Time |
|------|-----|------|-----|-----|------|
| EventBus (OBSERVER) | 🟢 Raw | 10.6 | 0.0 | 12 | 0.6s |
| EventBus (OBSERVER) | 🔵 GKG | 12.0 | 0.0 | 12 | 0.6s |
| HTTP Router (COMPOSITE+STRATEGY) | 🟢 Raw | 14.0 | 0.0 | 14 | 0.6s |
| HTTP Router (COMPOSITE+STRATEGY) | 🔵 GKG | 13.3 | 0.0 | 14 | 0.6s |
| UserRepository (REPOSITORY) | 🟢 Raw | 11.4 | 0.0 | 16 | 0.6s |
| UserRepository (REPOSITORY) | 🔵 GKG | 15.0 | 0.0 | 16 | 1.2s |
| Producer-Consumer (ownership) | 🟢 Raw | 15.0 | 0.0 | 15 | 0.9s |
| Producer-Consumer (ownership) | 🔵 GKG | 15.0 | 0.0 | 15 | 1.1s |

## 5 · Graph structure scores (GKG only)

Evaluates the architecture itself — level coverage (Macro/Meso/Micro), declared behaviors, IO contracts, validator health.

In [7]:
rows = ['| Task | Arm | Depth | Behaviors | Contracts | Populated | X-edges | Valid | Total |']
rows.append('|------|-----|-------|-----------|-----------|-----------|---------|-------|-------|')
for task in TASKS:
    for arm_id, arm_label in [('B_gkg', '🔵 GKG'), ('C_agent', '🟡 Agent')]:
        runs = [r for r in all_results if r.task_id == task.id and r.arm == arm_id]
        if not runs: continue
        gs = runs[0].graph_score
        if gs.max_total == 0: continue
        rows.append(f'| {task.title} | {arm_label} | {gs.level_depth}/3 | '
                    f'{"\u2713" if gs.meso_has_behaviors else "\u2717"} | '
                    f'{"\u2713" if gs.micro_has_contracts else "\u2717"} | '
                    f'{"\u2713" if gs.all_macros_populated else "\u2717"} | '
                    f'{"\u2713" if gs.cross_module_edges else "\u2717"} | '
                    f'{"\u2713" if gs.validation_clean else "\u2717"} | '
                    f'{gs.total:.0f}/{gs.max_total:.0f} |')
display(Markdown('\n'.join(rows)))


| Task | Depth | Behaviors | Contracts | Populated | X-edges | Valid | Total |
|------|-------|-----------|-----------|-----------|---------|-------|-------|
| EventBus (OBSERVER) | 3/3 | ✓ | ✓ | ✓ | ✗ | ✓ | 8/9 |
| HTTP Router (COMPOSITE+STRATEGY) | 3/3 | ✓ | ✓ | ✓ | ✓ | ✓ | 9/9 |
| UserRepository (REPOSITORY) | 3/3 | ✓ | ✓ | ✓ | ✓ | ✓ | 9/9 |
| Producer-Consumer (ownership) | 3/3 | ✓ | ✓ | ✓ | ✗ | ✓ | 8/9 |

## 6 · Side-by-side code

In [8]:
import html as html_mod

for task in TASKS:
    a_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'A_raw']
    b_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'B_gkg']
    c_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'C_agent']
    code_a = html_mod.escape(a_runs[0].code if a_runs else '(no result)')
    code_b = html_mod.escape(b_runs[0].code if b_runs else '(no result)')
    code_c = html_mod.escape(c_runs[0].code if c_runs else '(no result)')
    sc_a = a_runs[0].code_score if a_runs else ScoreCard()
    sc_b = b_runs[0].code_score if b_runs else ScoreCard()
    sc_c = c_runs[0].code_score if c_runs else ScoreCard()
    display(HTML(f'''
    <h3>{task.title}</h3>
    <div style="display:flex; gap:8px;">
      <div style="flex:1; border:1px solid #666; padding:8px; border-radius:6px;">
        <b>🟢 Raw LLM</b> ({sc_a.total:.1f}/{sc_a.max_total:.0f})<br>
        <pre style="background:#1a1a2e; color:#eee; padding:8px; border-radius:4px;
                    font-size:11px; white-space:pre-wrap;">{code_a}</pre></div>
      <div style="flex:1; border:1px solid #666; padding:8px; border-radius:6px;">
        <b>🔵 GKG hand-coded</b> ({sc_b.total:.1f}/{sc_b.max_total:.0f})<br>
        <pre style="background:#162447; color:#eee; padding:8px; border-radius:4px;
                    font-size:11px; white-space:pre-wrap;">{code_b}</pre></div>
      <div style="flex:1; border:1px solid #666; padding:8px; border-radius:6px;">
        <b>🟡 GKG agent-built</b> ({sc_c.total:.1f}/{sc_c.max_total:.0f})<br>
        <pre style="background:#1a2e1a; color:#eee; padding:8px; border-radius:4px;
                    font-size:11px; white-space:pre-wrap;">{code_c}</pre></div>
    </div>'''))


## 7 · Graph tree (GKG architecture)

Shows the hierarchy: Macro → Meso → Micro, with metadata at each level.

In [9]:
def print_tree(g: Graph, title: str):
    print(f'\n{"═"*60}')
    print(f'  {title}')
    print(f'  phase={g.phase.value}  nodes={len(g.nodes)}  edges={len(g.edges)}  violations={len(g.validate())}')
    print(f'{"═"*60}')
    roots = [n for n in g.nodes.values() if n.parent is None]

    def show(n, depth=0):
        pad = '  ' * depth
        level_icons = {Level.MACRO: '📦', Level.MESO: '🏛', Level.MICRO: '⚙'}
        icon = level_icons.get(n.level, '?')
        print(f'{pad}{icon} [{n.level.value}] {n.name}  ({n.status.value})')
        if isinstance(n.payload, MacroPayload):
            print(f'{pad}   lang={n.payload.language}  ownership={n.payload.ownership.value}')
        elif isinstance(n.payload, MesoPayload):
            pat = n.payload.design_pattern.value if n.payload.design_pattern != DesignPattern.PAT_NONE else '-'
            print(f'{pad}   pattern={pat}  behaviors={n.payload.behaviors}')
        elif isinstance(n.payload, MicroPayload):
            print(f'{pad}   in={n.payload.inputs}  out={n.payload.outputs}')
        for cid in n.children:
            show(g.nodes[cid], depth + 1)

    for r in roots:
        show(r)
    if g.edges:
        print(f'\n  Edges:')
        for e in g.edges.values():
            sn = g.nodes[e.src].name if e.src in g.nodes else '?'
            dn = g.nodes[e.dst].name if e.dst in g.nodes else '?'
            print(f'    {e.kind.value}: {sn} → {dn}  order={e.order}')

for task in TASKS:
    b_runs = [r for r in all_results if r.task_id == task.id and r.arm == 'B_gkg' and r.graph]
    if b_runs:
        print_tree(b_runs[0].graph, task.title)


════════════════════════════════════════════════════════════
  EventBus (OBSERVER)
  phase=DONE  nodes=6  edges=0  violations=0
════════════════════════════════════════════════════════════
📦 [MACRO] Events  (LAYER_STABLE)
   lang=python  ownership=SINGLE_WRITER
  🏛 [MESO] EventBus  (LAYER_STABLE)
     pattern=OBSERVER  behaviors=['subscribe', 'unsubscribe', 'notify', 'publish']
    ⚙ [MICRO] subscribe_impl  (CODED)
       in=['event_type', 'handler']  out=[]
    ⚙ [MICRO] unsubscribe_impl  (CODED)
       in=['event_type', 'handler']  out=[]
    ⚙ [MICRO] notify_impl  (CODED)
       in=['event_type', 'data']  out=[]
    ⚙ [MICRO] publish_impl  (CODED)
       in=['event_type', 'data']  out=[]

════════════════════════════════════════════════════════════
  HTTP Router (COMPOSITE+STRATEGY)
  phase=DONE  nodes=8  edges=2  violations=0
════════════════════════════════════════════════════════════
📦 [MACRO] RouterModule  (LAYER_STABLE)
   lang=python  ownership=SINGLE_WRITER
  🏛 [MESO] Router

## 8 · Aggregate verdict

In [10]:
print('CODE QUALITY')
print('\u2500' * 54)
arm_norms = {}
for arm, label in [('A_raw', '🟢 Raw'), ('B_gkg', '🔵 GKG hand-coded'), ('C_agent', '🟡 GKG agent-built')]:
    runs = [r for r in all_results if r.arm == arm]
    if not runs: continue
    norms = [r.code_score.total / max(r.code_score.max_total, 1) for r in runs]
    times = [r.elapsed for r in runs]
    arm_norms[arm] = norms
    print(f'{label}:  score={statistics.mean(norms):.1%} \u00b1 {statistics.pstdev(norms):.1%}  '
          f'time={statistics.mean(times):.1f}s')

a_norms = arm_norms.get('A_raw', [0])
b_norms = arm_norms.get('B_gkg', [0])
c_norms = arm_norms.get('C_agent', [0])
lift_b = statistics.mean(b_norms) - statistics.mean(a_norms)
lift_c = statistics.mean(c_norms) - statistics.mean(a_norms)
a_std = statistics.pstdev(a_norms) or 0.001
b_std = statistics.pstdev(b_norms) or 0.001
c_std = statistics.pstdev(c_norms) or 0.001
print(f'\n  GKG hand-coded lift:   {lift_b:+.1%}  (variance ratio {b_std/a_std:.2f})')
print(f'  GKG agent-built lift:  {lift_c:+.1%}  (variance ratio {c_std/a_std:.2f})')

a_t = statistics.mean([r.elapsed for r in all_results if r.arm=='A_raw'] or [0.01])
b_t = statistics.mean([r.elapsed for r in all_results if r.arm=='B_gkg'] or [0.01])
c_t = statistics.mean([r.elapsed for r in all_results if r.arm=='C_agent'] or [0.01])
print(f'  Cost ratio B/A: {b_t/a_t:.1f}x    C/A: {c_t/a_t:.1f}x')

print()
print('GRAPH STRUCTURE (GKG arms only)')
print('\u2500' * 54)
for arm, label in [('B_gkg', '🔵 GKG hand-coded'), ('C_agent', '🟡 GKG agent-built')]:
    b_gs = [r.graph_score for r in all_results if r.arm == arm and r.graph_score.max_total > 0]
    if not b_gs: continue
    gs_norms = [gs.total / gs.max_total for gs in b_gs]
    print(f'  {label}:')
    print(f'    graph score:        {statistics.mean(gs_norms):.0%} '
          f'({statistics.mean([gs.total for gs in b_gs]):.1f}/{b_gs[0].max_total:.0f})')
    print(f'    3-level coverage:   {statistics.mean([gs.level_depth for gs in b_gs]):.1f}/3')
    print(f'    all macros filled:  {statistics.mean([gs.all_macros_populated for gs in b_gs]):.0%}')
    print(f'    cross-module edges: {statistics.mean([gs.cross_module_edges for gs in b_gs]):.0%}')
    print(f'    validation clean:   {statistics.mean([gs.validation_clean for gs in b_gs]):.0%}')

print()
print('INTERPRETATION')
print('\u2500' * 54)
best_lift = max(lift_b, lift_c)
if best_lift > 0.02:
    print('GKG blueprint context produces better code.')
elif best_lift < -0.02:
    print('Raw LLM wins on code quality.')
else:
    print('Code quality comparable. GKG value is in architecture.')
if b_std / a_std < 0.7:
    print('Hand-coded GKG significantly more consistent than raw.')
if c_norms and c_std / a_std < 0.7:
    print('Agent-built GKG significantly more consistent than raw.')
print()
print('KEY INSIGHT: compare B vs C graph scores to measure architect agent quality.')
print('If C graph score < B, architect LLM fails to build full blueprints.')
print('If C code score ~ B, GKG is robust to blueprint imperfection.')


CODE QUALITY
──────────────────────────────────────────────────
🟢 Raw:  score=89.9% ± 11.7%  time=0.7s
🔵 GKG:  score=97.2% ± 2.8%  time=0.9s

  Quality lift:   +7.3%
  Variance ratio: 0.24
  Cost ratio:     1.3x

GRAPH STRUCTURE (GKG only)
──────────────────────────────────────────────────
  graph score:        94% (8.5/9)
  3-level coverage:   3.0/3
  all macros filled:  100%
  cross-module edges: 50%
  validation clean:   100%

INTERPRETATION
──────────────────────────────────────────────────
GKG blueprint context produces better code.
GKG significantly more consistent.

KEY INSIGHT: Raw LLM has no architecture. GKG gives you a typed, validated,
inspectable graph BEFORE any code exists. The graph IS the deliverable at
Macro/Meso/Micro level.
